In [4]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sixs_tools import read_nxs5 as rn5
from scipy.signal import savgol_filter

In [5]:
import larch
from larch import Group
from larch.xafs import pre_edge

In [ ]:
import json
from pathlib import Path
with open(Path.cwd().parent.parent / 'config.json') as f:
    cfg = json.load(f)
homePath = Path(cfg['data_root'])
mask = np.load(homePath / cfg['mask_path'])

In [10]:
XASdict = {'Sample2':'Sample2',
           '287':{'edge':'Ni_K', 'trim':[2,0], 'label':'N2_RT',
                 'normParams':{'pre1':-100, 'pre2':-25, 'norm1':10, 'norm2':200, 'nnorm':1, 'nvict':4}},
           '289':{'edge':'Co_K', 'trim':[1,0], 'label':'N2_RT',
                 'normParams':{'pre1':-100, 'pre2':-25, 'norm1':10, 'norm2':200, 'nnorm':1, 'nvict':5}},
           '329':{'edge':'Ni_K', 'trim':[2,0], 'label':'H2_400C',
                 'normParams':{'pre1':-100, 'pre2':-25, 'norm1':10, 'norm2':200, 'nnorm':2, 'nvict':-1}},
           '331':{'edge':'Co_K', 'trim':[1,0], 'label':'H2_400C',
                 'normParams':{'pre1':-100, 'pre2':-20, 'norm1':10, 'norm2':200, 'nnorm':2, 'nvict':3}},
           '366':{'edge':'Ni_K', 'trim':[2,0], 'label':'Mix_300C',
                 'normParams':{'pre1':-100, 'pre2':-20, 'norm1':10, 'norm2':200, 'nnorm':1, 'nvict':4}},
           '368':{'edge':'Co_K', 'trim':[1,0], 'label':'Mix_300C',
                 'normParams':{'pre1':-100, 'pre2':-20, 'norm1':10, 'norm2':200, 'nnorm':1, 'nvict':3}},
           '384':{'edge':'Ni_K', 'trim':[2,0], 'label':'Mix_400C', 
                 'normParams':{'pre1':-40, 'pre2':-15, 'norm1':10, 'norm2':200, 'nnorm':1, 'nvict':6}},
           '386':{'edge':'Co_K', 'trim':[1,0], 'label':'Mix_400C', 
                  'normParams':{'pre1':-100, 'pre2':-20, 'norm1':10, 'norm2':200, 'nnorm':1, 'nvict':4}},
           
}

nxs = 289
path = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
path.exists()

True

In [11]:
nxs = 287

# load whole nxs
filePath = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
spectrum = rn5.DataSet(filePath.name, filePath.parent)

# left and right trims are defined in the XASdict
trimL, trimR = XASdict[str(nxs)]['trim'][0], -1 -XASdict[str(nxs)]['trim'][1]

# extract trimmed data 
energy = spectrum.energy[trimL:trimR]
xbpm = spectrum.xbpm_cx1_i[trimL:trimR]
img_masked = spectrum.xpad140*(1-mask)

# integrate whole images then trim
roi0_masked = np.sum(img_masked, axis=(1, 2))[trimL:trimR]

# calculate absorption
XAS_masked = np.log(roi0_masked/xbpm)

label = f'#{nxs} : {XASdict[str(nxs)]['edge']} - {XASdict[str(nxs)]['label']}'

plt.figure(figsize=(5,3))
plt.plot(energy, XAS_masked, label = label)
plt.legend()
plt.show()

TypeError: 'NoneType' object is not iterable

In [ ]:
# instance larch object with energy and signal
data = Group(name='my group', energy=energy*1000, mu=XAS_masked)

normParams = {'pre1':-100, # the lower bond of the function to fit the pre-edge
         'pre2':-20, # the upper bond of the function to fit the pre-edge
         'norm1':10, # the lower bond of the function to fit the post-edge
         'norm2':200, # the upper bond of the function to fit the post-edge
         'nnorm':1, # degree for the polynomials
         'nvict':4}

In [ ]:
# tweak params, then copy it in the XASdict
normParams = {'pre1':-100, 'pre2':-25, 'norm1':10, 'norm2':200, 'nnorm':1, 'nvict':4}

pre_edge(data.energy, data.mu, group=data, **normParams)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(data.energy, data.mu, label='XAS')
axes[0].plot(data.energy, data.pre_edge,"b--")
axes[0].plot(data.energy, data.post_edge,"g--")
axes[0].set_xlabel("Energy (keV)")
axes[0].set_title("XAS : log(roi0/mon1)")

axes[1].plot(data.energy, data.norm, label='XAS')
#plt.plot(data.energy, data.xas, label='XAS')
axes[1].set_xlabel("Energy (keV)")
axes[1].set_title("Normalised $\mu(E)$")
plt.show()

In [ ]:
'386'

In [ ]:
nxs = 287

# load whole nxs
filePath = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
spectrum = rn5.DataSet(filePath.name, filePath.parent)

# left and right trims are defined in the XASdict
trimL, trimR = XASdict[str(nxs)]['trim'][0], -1 -XASdict[str(nxs)]['trim'][1]

# extract trimmed data 
energy = spectrum.energy[trimL:trimR]
mon1 = spectrum.xbpm_cx1_i[trimL:trimR]
img_masked = spectrum.xpad140*(1-mask)

# integrate whole images then trim
roi0_masked = np.sum(img_masked, axis=(1, 2))[trimL:trimR]

# calculate absorption
XAS_masked = np.log(roi0_masked/mon1)

# instance larch object with energy and signal
data = Group(name='my group', energy=energy*1000, mu=XAS_masked)

# normalize spectrum using larch function and predetermined set of params save in XASdict
pre_edge(data.energy, data.mu, group=data, **XASdict[str(nxs)]['normParams'])

data

In [ ]:
# plot raw and normaliezed data

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].plot(data.energy, data.mu, label='XAS')
axes[0].plot(data.energy, data.pre_edge,"b--")
axes[0].plot(data.energy, data.post_edge,"g--")
axes[0].set_xlabel("Energy (keV)")
axes[0].set_title("Raw log(roi0/mon1)")

axes[1].plot(data.energy, data.norm, label='XAS')
#plt.plot(data.energy, data.xas, label='XAS')
axes[1].set_xlabel("Energy (keV)")
axes[1].set_title("Normalised $\mu(E)$")
plt.show()

In [ ]:
#NXStoPlot = [289, 331, 368, 386] # Co
NXStoPlot = [287,  329, 366, 384] # Ni

#fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for nxs in NXStoPlot:
    filePath = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
    spectrum = rn5.DataSet(filePath.name, filePath.parent)
    trimL, trimR = XASdict[str(nxs)]['trim'][0], -1 -XASdict[str(nxs)]['trim'][1]
    
    energy = spectrum.energy[trimL:trimR]
    mon1 = spectrum.xbpm_cx1_i[trimL:trimR]
    img_masked = spectrum.xpad140*(1-mask)
    roi0_masked = np.sum(img_masked, axis=(1, 2))[trimL:trimR]
    XAS_masked = np.log(roi0_masked/mon1)
    
    data = Group(name='my group', energy=energy*1000, mu=XAS_masked)
    pre_edge(data.energy, data.mu, group=data, **XASdict[str(nxs)]['normParams'])

    label = f'#{nxs} : {XASdict[str(nxs)]['edge']} - {XASdict[str(nxs)]['label']}'

    plt.plot(data.energy, data.flat, label=label)

plt.legend()

In [ ]:
zip?

In [ ]:

NXStoPlot = [289, 287, 329, 331, 366, 368, 384, 386]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for nxs in NXStoPlot:
    
    filePath = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
    spectrum = rn5.DataSet(filePath.name, filePath.parent)
    trimL, trimR = XASdict[str(nxs)]['trim'][0], -1 -XASdict[str(nxs)]['trim'][1]
    
    energy = spectrum.energy[trimL:trimR]
    mon1 = spectrum.xbpm_cx1_i[trimL:trimR]
    img_masked = spectrum.xpad140*(1-mask)
    roi0_masked = np.sum(img_masked, axis=(1, 2))[trimL:trimR]
    XAS_masked = np.log(roi0_masked/mon1)
    
    data = Group(name='my group', energy=energy*1000, mu=XAS_masked)
    pre_edge(data.energy, data.mu, group=data, **XASdict[str(nxs)]['normParams'])

    label = f'#{nxs} : {XASdict[str(nxs)]['label']} ; e0 = {data.e0:.1f} ; ES = {data.edge_step:.3g}'
    
    if XASdict[str(nxs)]['edge'] == 'Ni_K':
        axes[0].plot(data.energy, data.flat, label=label)
    elif XASdict[str(nxs)]['edge'] == 'Co_K':
        axes[1].plot(data.energy, data.flat, label=label)

axes[0].legend(fontsize = 'small')
axes[0].set_xlabel('Energy (eV)')
axes[0].set_ylabel('Normalised $\mu(E)$')
axes[0].set_title("Ni K edge")
axes[1].legend(fontsize = 'small')
axes[1].set_xlabel('Energy (eV)')
axes[1].set_title("Co K edge")

plt.tight_layout()

In [ ]:
flagSave = False


NXStoPlot = [83, 141, 228, 253, 287] # Ni
NXStoPlot = [83, 141, 228, 253, 287] # Ni
#NXStoPlot = [89, 144, 232, 255] # Co
#NXStoPlot = [100, 147, 236, 257] # Fe


for nxs in NXStoPlot:
    filePath = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
    spectrum = rn5.DataSet(filePath.name, filePath.parent)
    energy = spectrum.energy
    mon0 = spectrum.imon0
    mon1 = spectrum.xbpm_cx1_i
    
    roi0 = np.sum(spectrum.xpad140, axis=(1, 2))
    roi0_masked = np.sum(spectrum.xpad140*(1-mask), axis=(1, 2))
    XAS = np.log(roi0/mon1)
    XAS_masked = np.log(roi0_masked/mon1)
    trimL, trimR = XASdict[str(nxs)]['trim'][0], -1 -XASdict[str(nxs)]['trim'][1]
    
    if flagSave:
        energy_trimmed, XAS_trimmed = energy[trimL:trimR], XAS_masked[trimL:trimR]
        data_to_save = np.column_stack((energy_trimmed, XAS_trimmed))
        output_filename = f"{nxs}_{filePath.name.split('_')[0]}_{XASdict[str(nxs)]['edge']}.txt" 
        np.savetxt(output_filename, data_to_save, header="Energy\tXAS", comments='', delimiter='\t')
    
    label = f'#{nxs} : {XASdict[str(nxs)]['edge']} - {XASdict[str(nxs)]['label']}'
    plt.plot(energy[trimL:trimR], XAS_masked[trimL:trimR], label = label)

plt.legend()

In [ ]:
%matplotlib inline
plt.close()

NXStoPlot = [287, 329, 366] # Ni sample2
#NXStoPlot = [289, 331, 368] # Co sample2
#NXStoPlot = [83, 141, 228, 253, 287] # Ni
NXStoPlot = [89, 144, 232, 255] # Co
#NXStoPlot = [100, 147, 236] # Fe

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for nxs in NXStoPlot:
    filePath = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
    spectrum = rn5.DataSet(filePath.name, filePath.parent)
    energy = spectrum.energy
    mon0 = spectrum.imon0
    mon1 = spectrum.xbpm_cx1_i
    mon1_filt = savgol_filter(mon1, window_length=13, polyorder=3)
    gap = spectrum.gap
    bragg = spectrum.bragg

    roi0 = np.sum(spectrum.xpad140, axis=(1, 2))
    roi0_masked = np.sum(spectrum.xpad140*(1-mask), axis=(1, 2))
    XAS = np.log(roi0/mon1)
    XAS_masked = np.log(roi0_masked/mon1)
    
    XAS_mon = np.log(mon1/mon0)
    
    trimL, trimR = XASdict[str(nxs)]['trim'][0], -1 -XASdict[str(nxs)]['trim'][1]
    label = f'#{nxs} : {XASdict[str(nxs)]['edge']} - {XASdict[str(nxs)]['label']}'
    axes[0].plot(energy[trimL:trimR], XAS_masked[trimL:trimR], label = label)
    axes[0].legend()
    axes[0].set_xlabel("Energy (keV)")
    axes[0].set_title("XAS : log(roi0/mon1)")
    
    axes[1].plot(energy[trimL:trimR], mon1[trimL:trimR])
    axes[1].set_title("Mon1 : xbpm_cx1_i")
    axes[1].set_xlabel("Energy (keV)")

plt.tight_layout()

In [ ]:
6.482600+0.02


In [ ]:

21.5*np.sqrt(12)/np.sqrt(4)

In [ ]:
1500*0.3

In [ ]:
SBS.a2scan(bragg, 13.784, 13.373, gap, 6.333, 6.503, 250, 2)


In [ ]:
from numpy.polynomial import Polynomial
pitch = [16.61, 19.44, 19.69, 20.1, 20.39, 20.97, 21.35, 25.7]

energ = [4.9, 7.06, 7.3, 7.65, 7.9, 8.3, 8.55, 18.44]

line = Polynomial.fit(energ[:-1], pitch[:-1], deg=1, domain=[])
intercept, slope = line.coef
y2= 4.9*slope+intercept
print(y2)

plt.plot(energ[:-1], pitch[:-1], 'r.')


In [ ]:
filePath = [file_path for file_path in homePath.rglob(f"*{nxs}.nxs")][0]
trimL, trimR = XASdict[str(nxs)]['trim'][0], -1 -XASdict[str(nxs)]['trim'][1]
spectrum = rn5.DataSet(filePath.name, filePath.parent)

energy = spectrum.energy[trimL:trimR]
mon0 = spectrum.imon0[trimL:trimR]
mon1 = spectrum.xbpm_cx1_i[trimL:trimR]
roi0 = np.sum(spectrum.xpad140, axis=(1, 2))[trimL:trimR]
roi0_masked = np.sum(spectrum.xpad140*(1-mask), axis=(1, 2))[trimL:trimR]

XAS = np.log(roi0/mon1)
XAS_masked = np.log(roi0_masked/mon1)

spectrum.attlist